# 🎯 Hull Tactical Market Prediction - Inference Server

## ⚠️ IMPORTANT: This notebook MUST run on Kaggle platform!

**This is an INFERENCE COMPETITION notebook.**
- ❌ Cannot run locally (kaggle_evaluation module not available)
- ✅ Must upload and run on Kaggle notebook environment
- ✅ Requires Kaggle's inference competition infrastructure

## 📌 Setup Instructions

### Step 1: Create ZIP file (Already done! ✅)

```bash
bash create_kaggle_zip.sh
```

This creates `prediction_market_modules.zip` containing:
- `src/` - All Python modules (data, features, models, etc.)
- `conf/params.yaml` - Configuration file

### Step 2: Upload Dataset to Kaggle

1. **Go to**: https://www.kaggle.com/datasets
2. **Click**: "New Dataset"
3. **Upload**: `prediction_market_modules.zip`
4. **Name it**: 원하는 이름 (예: `prediction-market-modules`)
5. **Click**: "Create"

### Step 3: Create New Notebook on Kaggle

1. **Go to competition**: https://www.kaggle.com/competitions/hull-tactical-market-prediction
2. **Click**: "Code" → "New Notebook"
3. **Copy this entire notebook** into the new Kaggle notebook

### Step 4: Configure Dataset Name

**⚠️ IMPORTANT**: 첫 번째 코드 셀에서 데이터셋 이름을 수정하세요!

```python
# ========== CONFIGURATION: 데이터셋 이름 (여기만 수정하세요!) ==========
DATASET_NAME = "your-dataset-name"  # Kaggle에 업로드한 이름으로 변경
# ====================================================================
```

### Step 5: Add Dataset to Notebook

1. **In Kaggle Notebook**: Click "Add Data" (right panel)
2. **Your Datasets** → Find your uploaded dataset
3. **Click**: "Add"

### Step 6: Run Notebook

1. **Click**: "Run All"
2. **Wait**: ~15 minutes for training to complete
3. **Submit**: Click "Submit to Competition"

## 🚀 What This Notebook Does:

### Training Phase (runs once):
1. ✅ Loads and preprocesses training data
2. ✅ Engineers 600+ features → selects best ~150
3. ✅ Trains LightGBM model with cross-validation
4. ✅ Stores model in memory

### Inference Phase (for each timestep):
1. ✅ Receives new market data from Kaggle
2. ✅ Applies same preprocessing + feature engineering
3. ✅ Predicts return using trained model
4. ✅ Converts to allocation (0.0 to 2.0)
5. ✅ Returns single float within 5-minute limit

## 📊 Competition Details:

- **Prediction**: Single allocation value (0 = no investment, 1 = full market, 2 = 2x leverage)
- **Response Time**: 5 minutes per prediction
- **Startup Time**: 15 minutes for initial training
- **Evaluation**: Timestep-by-timestep (streaming)


## 1️⃣ Setup Module Paths

In [ ]:
import os
import sys
from pathlib import Path
import warnings
import logging

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ========== LOGGING CONFIGURATION ==========
# Disable ALL logging (no log files, no console logs)
logging.disable(logging.CRITICAL)  # Disable all logging

# Remove all handlers from root logger
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Suppress specific loggers
for logger_name in ['src.data', 'src.features', 'src.models', 'src.risk', 'src.position', 
                    'lightgbm', 'prediction_market', 'matplotlib', 'PIL']:
    logger = logging.getLogger(logger_name)
    logger.disabled = True
    logger.propagate = False
    logger.handlers = []

# Only show notebook-level progress
print("="*80)
print("SETTING UP MODULE PATHS")
print("="*80)

# ========== CONFIGURATION: 데이터셋 이름 (여기만 수정하세요!) ==========
DATASET_NAME = "prediction-market-modules"
# ====================================================================

# Add kaggle_evaluation to path (from competition data)
kaggle_eval_path = Path("/kaggle/input/hull-tactical-market-prediction")
if kaggle_eval_path.exists():
    sys.path.insert(0, str(kaggle_eval_path))
    print(f"✓ Added kaggle_evaluation path")

# Add custom modules to path - try multiple possible locations
dataset_locations = [
    Path(f"/kaggle/input/{DATASET_NAME}"),
    Path(f"/kaggle/input/{DATASET_NAME}/prediction_market_modules"),
]

module_found = False
for dataset_dir in dataset_locations:
    if dataset_dir.exists():
        sys.path.insert(0, str(dataset_dir))
        
        # Check if src directory exists
        src_dir = dataset_dir / "src"
        if src_dir.exists():
            print(f"✓ Found src directory")
            module_found = True
            # Save for later use
            globals()['DATASET_PATH'] = str(dataset_dir)
            break

if not module_found:
    print(f"\n⚠️  src module not found. Available files in /kaggle/input/:")
    input_dir = Path("/kaggle/input/")
    if input_dir.exists():
        for item in input_dir.iterdir():
            print(f"  📁 {item.name}")

print(f"\n✅ Path setup complete!")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Logging: Minimal (notebook-level only)")

## 2️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

# Try to import kaggle_evaluation (only available in Kaggle environment)
try:
    import kaggle_evaluation.default_inference_server
    print("✓ kaggle_evaluation imported")
    KAGGLE_ENV = True
except ImportError:
    print("⚠️  kaggle_evaluation not found - this is normal if running locally")
    print("   This notebook MUST be run on Kaggle platform for submission")
    KAGGLE_ENV = False

print("="*80)
print("IMPORTING MODULES")
print("="*80)

# Import modules with error handling
DataLoader = None
FeatureEngineering = None
ReturnPredictor = None
RiskForecaster = None
QuantileBinningMapper = None
load_config = None
Timer = None

try:
    from src.data import DataLoader
    print("✓ DataLoader imported")
except Exception as e:
    print(f"❌ DataLoader import failed: {e}")
    raise

try:
    from src.features import FeatureEngineering
    print("✓ FeatureEngineering imported")
except Exception as e:
    print(f"❌ FeatureEngineering import failed: {e}")
    raise

try:
    from src.models import ReturnPredictor
    print("✓ ReturnPredictor imported")
except Exception as e:
    print(f"❌ ReturnPredictor import failed: {e}")
    raise

try:
    from src.risk import RiskForecaster
    print("✓ RiskForecaster imported")
except Exception as e:
    print(f"❌ RiskForecaster import failed: {e}")
    raise

try:
    from src.position import QuantileBinningMapper
    print("✓ QuantileBinningMapper imported")
except Exception as e:
    print(f"❌ QuantileBinningMapper import failed: {e}")
    raise

try:
    from src.utils import load_config, Timer
    print("✓ Utils imported")
except Exception as e:
    print(f"❌ Utils import failed: {e}")
    raise

print("\n✅ All imports successful!")

if not KAGGLE_ENV:
    print("\n" + "="*80)
    print("⚠️  WARNING: Not running in Kaggle environment!")
    print("="*80)
    print("This notebook requires Kaggle's inference competition environment.")
    print("Please upload and run this notebook on Kaggle platform.")
    print("="*80)

## 3️⃣ Load and Train Model (Once)

This cell trains the model and stores it in global variables.
The model will be loaded once and reused for all predictions.

In [ ]:
# ==================== CONFIGURATION ====================
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Kaggle dataset path
DATASET_NAME = "prediction-market-modules"
BASE_PATH = f"/kaggle/input/{DATASET_NAME}"

# Add modules to path
sys.path.insert(0, BASE_PATH)

# Import after path is set
import numpy as np
import pandas as pd
from pathlib import Path

# Import custom modules
from src.data import DataProcessor
from src.features import FeatureEngineering
from src.models import ReturnPredictor
from src.position import QuantileBinningMapper
from src.risk import RiskForecaster

print("✓ All modules imported successfully!")

# ==================== STEP 1: Load and Process Data ====================
print("\n" + "="*80)
print("TRAINING PIPELINE")
print("="*80)

print("\n[1/8] Loading and processing training data...")
config_path = f"{BASE_PATH}/conf/params.yaml"

processor = DataProcessor(config_path=config_path)
train_df = processor.load_and_process(
    f"{BASE_PATH}/data/raw/train.csv",
    is_train=True,
    add_target=True,
    scale=True,
    scale_method='robust',
    window=60
)
print(f"      ✓ Data loaded and preprocessed: {train_df.shape}")

# ==================== STEP 2: Feature Engineering ====================
print("\n[2/8] Engineering features...")
fe = FeatureEngineering(config_path=config_path)
train_features = fe.fit_transform(train_df)
print(f"      ✓ Features created: {train_features.shape}")

# ==================== STEP 3: Feature Selection for Return Model (WEIGHTED ENSEMBLE) ====================
print("\n[3/8] Selecting features for return model (Weighted Ensemble)...")
train_selected, selected_features, score_summary = fe.select_features_weighted_ensemble(
    train_features,
    target_col='forward_returns',
    weights={
        'correlation': 0.25,
        'mutual_info': 0.55,  # 금융 데이터는 비선형 관계가 많음
        'variance': 0.20
    },
    top_n=150  # 150개로 늘림 (correlation 제거 후 ~120개 남을 것)
)

train_selected, _ = fe.remove_correlated_features(
    train_selected,
    threshold=0.95,
    target_col='forward_returns'
)

FEATURE_COLS = [
    col for col in train_selected.columns 
    if col not in ['date_id', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
]
print(f"      ✓ Selected {len(FEATURE_COLS)} features (weighted ensemble)")

# ==================== STEP 4: Train Return Model ====================
print("\n[4/8] Training return prediction model...")
print("      (This takes ~2-3 minutes with 5-fold CV)")
predictor = ReturnPredictor(model_type='lightgbm', config_path=config_path)
results = predictor.train_cv(
    df=train_selected,
    target_col='forward_returns',
    date_col='date_id'
)
print(f"      ✓ Return model trained (OOF Score: {results['oof_score']:.6f})")

# ==================== STEP 5: Create Risk Labels ====================
print("\n[5/8] Creating risk labels...")
from src.risk import RiskLabeler

risk_labeler = RiskLabeler(config_path=config_path)
train_features_risk = risk_labeler.fit_transform(train_features, target_col='forward_returns')
n_valid_risk = train_features_risk['risk_label'].notna().sum()
print(f"      ✓ Risk labels created: {n_valid_risk} valid samples")

# ==================== STEP 6: Feature Selection for Risk Model (WEIGHTED ENSEMBLE) ====================
print("\n[6/8] Selecting features for risk model (Weighted Ensemble)...")
train_risk_selected, risk_features, risk_score_summary = fe.select_features_weighted_ensemble(
    train_features_risk,
    target_col='risk_label',
    weights={
        'correlation': 0.30,
        'mutual_info': 0.50,  # 비선형 관계 중요
        'variance': 0.20
    },
    top_n=150
)

train_risk_selected, _ = fe.remove_correlated_features(
    train_risk_selected,
    threshold=0.95,
    target_col='risk_label'
)

RISK_FEATURE_COLS = [
    col for col in train_risk_selected.columns 
    if col not in ['date_id', 'risk_label', 'forward_returns', 'risk_free_rate', 
                   'market_forward_excess_returns']
]
print(f"      ✓ Selected {len(RISK_FEATURE_COLS)} features (weighted ensemble)")

# ==================== STEP 7: Train Risk Model ====================
print("\n[7/8] Training risk forecasting model...")
print("      (This takes ~2-3 minutes with 5-fold CV)")
risk_forecaster = RiskForecaster(model_type='lightgbm', config_path=config_path)

# Train risk model and get OOF predictions
oof_risk, risk_models = risk_forecaster.train(
    df=train_risk_selected,
    feature_cols=RISK_FEATURE_COLS,
    risk_col='risk_label',
    n_folds=5
)

print(f"      ✓ Risk model trained")
print(f"      ✓ OOF risk predictions: min={oof_risk.min():.6f}, max={oof_risk.max():.6f}")

# ==================== STEP 8: Fit Position Mapper ====================
print("\n[8/8] Fitting position mapper on OOF predictions...")

position_mapper = QuantileBinningMapper(config_path=config_path)

# Use OOF predictions from return model
oof_returns_full = results['oof_predictions']
oof_risk_full = oof_risk.copy()

# Filter out NaN values for position mapper fitting
valid_idx = ~np.isnan(oof_risk) & ~np.isnan(train_risk_selected['risk_label'].values)
min_len = min(len(oof_returns_full), len(oof_risk_full))
oof_returns_full = oof_returns_full[:min_len]
oof_risk_full = oof_risk_full[:min_len]

# Create valid mapper index (both return and risk must be valid)
valid_mapper_idx = ~np.isnan(oof_returns_full) & ~np.isnan(oof_risk_full)
oof_returns_valid = oof_returns_full[valid_mapper_idx]
oof_risk_valid = oof_risk_full[valid_mapper_idx]

# Fit position mapper
position_mapper.fit(oof_returns_valid, oof_risk_valid)
print(f"      ✓ Position mapper fitted on {len(oof_returns_valid)} valid samples")
print(f"      ✓ Bin edges: {position_mapper.bin_edges}")

print("\n" + "="*80)
print("✓ TRAINING COMPLETE!")
print("="*80)
print(f"\nModel Summary:")
print(f"  - Return Model OOF Score: {results['oof_score']:.6f}")
print(f"  - Features (Return): {len(FEATURE_COLS)}")
print(f"  - Features (Risk): {len(RISK_FEATURE_COLS)}")
print(f"  - Position Mapper: Fitted")
print(f"  - Feature Selection: Weighted Ensemble (correlation=0.25, MI=0.55, variance=0.20)")


## 4️⃣ Define Prediction Function

This function will be called for each timestep.
It receives a Polars DataFrame and must return a single float value.

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """
    Predict allocation using Return and Risk models with Quantile Binning strategy.
    
    Args:
        test: Polars DataFrame with test data for current timestep
        
    Returns:
        float: Optimal allocation (0.0 to 2.0)
    """
    # Convert Polars to Pandas
    test_df = test.to_pandas()
    
    # DEBUG: Check input
    print(f"\n[DEBUG] Input shape: {test_df.shape}, columns: {len(test_df.columns)}")
    
    # Feature engineering (skip preprocessing - single row can't compute regime features)
    test_features = fe.transform(test_df)
    print(f"[DEBUG] After FE: {test_features.shape} features")
    
    # Add missing features with default value 0
    missing_count = 0
    for col in FEATURE_COLS:
        if col not in test_features.columns:
            test_features[col] = 0.0
            missing_count += 1
    
    for col in RISK_FEATURE_COLS:
        if col not in test_features.columns:
            test_features[col] = 0.0
            missing_count += 1
    
    print(f"[DEBUG] Missing features filled: {missing_count}")
    
    # ===== Return Prediction =====
    X_return = test_features[FEATURE_COLS]
    print(f"[DEBUG] X_return stats: mean={X_return.mean().mean():.4f}, std={X_return.std().mean():.4f}")
    
    r_hat = predictor.predict(X_return)
    print(f"[DEBUG] r_hat (return): {r_hat[0]:.6f}")
    
    # ===== Risk Prediction =====
    X_risk = test_features[RISK_FEATURE_COLS]
    print(f"[DEBUG] X_risk stats: mean={X_risk.mean().mean():.4f}, std={X_risk.std().mean():.4f}")
    
    sigma_hat = risk_forecaster.predict(X_risk)
    print(f"[DEBUG] sigma_hat (risk): {sigma_hat[0]:.6f}")
    
    # ===== Position Mapping using QuantileBinningMapper =====
    # Calculate z-score
    eps = 1e-6
    z_score = r_hat[0] / (sigma_hat[0] + eps)
    print(f"[DEBUG] z-score: {z_score:.6f}")
    print(f"[DEBUG] Bin edges: {position_mapper.bin_edges}")
    
    allocation = position_mapper.map_positions(r_hat, sigma_hat)[0]
    print(f"[DEBUG] allocation (before clip): {allocation:.6f}")
    
    # Safety clip to valid range
    allocation = np.clip(allocation, 0.0, 2.0)
    print(f"[DEBUG] allocation (final): {allocation:.6f}")
    
    return float(allocation)

print("✅ predict() function defined")
print(f"   Uses pre-trained models from Cell 6")
print(f"   Note: Single-row test data - regime features will be set to 0")
print(f"   Return features: {len(FEATURE_COLS)}")
print(f"   Risk features: {len(RISK_FEATURE_COLS)}")
print(f"   Position strategy: Quantile Binning ({position_mapper.n_bins} bins)")
print("\n⚠️  DEBUG MODE ENABLED - Will print detailed logs for each prediction")

## 5️⃣ Initialize Inference Server

In [ ]:
print("="*80)
print("INITIALIZING INFERENCE SERVER")
print("="*80)

if not KAGGLE_ENV:
    print("\n⚠️  Skipping server initialization (not in Kaggle environment)")
    print("Please run this notebook on Kaggle platform")
else:
    # Create inference server with our predict function
    inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
    
    print("\n✅ Inference server initialized!")
    print("\nServer configuration:")
    print("  - Prediction function: predict()")
    print("  - Response time limit: 5 minutes per prediction")
    print("  - Server startup limit: 15 minutes")
    print("\n⚡ Ready to serve predictions!")

## 6️⃣ Start Server (Kaggle Evaluation Mode)

**This cell starts the actual server when running on Kaggle's evaluation system.**

- **Kaggle mode**: Server waits for timestep-by-timestep data from Kaggle evaluation
- **Local mode**: Generates sample predictions using `predict()` function for testing

**⚠️ IMPORTANT**: Cell 13 below is REMOVED (was duplicate of this cell)

In [ ]:
# ==================== GENERATE SUBMISSION FILE ====================
print("="*80)
print("GENERATING SUBMISSION FILE")
print("="*80)

# Load test data
test_path = "/kaggle/input/hull-tactical-market-prediction/test.csv"
if not Path(test_path).exists():
    raise FileNotFoundError(f"Test data not found: {test_path}")

test_df = pd.read_csv(test_path)
print(f"\n✓ Test data loaded: {test_df.shape}")

# Generate predictions for ALL test samples
print("\nGenerating predictions for all test samples...")
predictions = []

for idx in range(len(test_df)):
    test_row = test_df.iloc[[idx]]
    test_pl = pl.from_pandas(test_row)
    
    # Use the predict() function
    allocation = predict(test_pl)
    predictions.append(allocation)
    
    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{len(test_df)} samples...")

print(f"\n✓ Predictions generated: {len(predictions)} samples")

# Create submission DataFrame
submission = pd.DataFrame({
    'date_id': test_df['date_id'].astype('int64'),
    'allocation': np.array(predictions).astype('float64')
})

# Clip to valid range
submission['allocation'] = submission['allocation'].clip(0, 2)

# Validation
print("\n" + "="*80)
print("VALIDATING SUBMISSION")
print("="*80)

assert list(submission.columns) == ['date_id', 'allocation'], "❌ Wrong column names!"
assert submission['allocation'].isna().sum() == 0, "❌ Contains NaN values!"
assert (submission['allocation'] >= 0).all(), "❌ Contains values < 0!"
assert (submission['allocation'] <= 2).all(), "❌ Contains values > 2!"

print("✓ Column names: ['date_id', 'allocation']")
print(f"✓ No missing values: {submission['allocation'].isna().sum()} NaN")
print(f"✓ Valid range: [{submission['allocation'].min():.4f}, {submission['allocation'].max():.4f}]")
print(f"✓ Mean allocation: {submission['allocation'].mean():.4f}")

# Save as Parquet
output_path = '/kaggle/working/submission.parquet'
submission.to_parquet(
    output_path,
    index=False,
    engine='pyarrow'
)

# Preview submission
print("\n" + "="*80)
print("SUBMISSION PREVIEW")
print("="*80)
print(submission.head(10))
print(f"\nAllocation distribution:")
print(submission['allocation'].value_counts().sort_index())
print(f"\nStats:")
print(submission['allocation'].describe())

print(f"\n✅ Submission saved to: {output_path}")
print(f"   Size: {len(submission)} rows")

# ==================== START INFERENCE SERVER (IF KAGGLE ENV) ====================
if KAGGLE_ENV:
    print("\n" + "="*80)
    print("STARTING INFERENCE SERVER")
    print("="*80)
    print("\n⚡ Server is now listening for real-time inference requests...")
    print("   (Submission file already created above)")
    print("   Waiting for timestep-by-timestep data from Kaggle...\n")
    
    # Start the inference server (this will block)
    inference_server.serve()
else:
    print("\n" + "="*80)
    print("🎉 READY FOR SUBMISSION!")
    print("="*80)

## 📊 Pipeline Summary

### Training Pipeline (Cell 6 - Runs Once on Startup):

```
[1/8] Load Data (train.csv)
   ↓
[2/8] Feature Engineering (600+ features)
   ↓
[3/8] Feature Selection - Return Model (~150 features)
   ↓
[4/8] Train Return Model (LightGBM, 5-fold CV) ⏳ 2-3 min
   ↓
[5/8] Create Risk Labels (future volatility)
   ↓
[6/8] Feature Selection - Risk Model (~150 features)
   ↓
[7/8] Train Risk Model (LightGBM, 5-fold CV) ⏳ 2-3 min
   ↓
[8/8] Initialize Position Mapper (Quantile Binning)
   ↓
✅ Ready for Inference! Total: ~5-10 minutes
```

### Inference Pipeline (Cell 7 predict() - Per Timestep):

```
New Data (test row)
   ↓
Feature Engineering (reuse fitted fe)
   ↓
Return Prediction (r_hat)
   ↓
Risk Prediction (sigma_hat)
   ↓
Position Mapping (QuantileBinningMapper)
   ↓
Return Single Float ✅
```

### Key Components:

| Component | What It Does | Cell |
|-----------|--------------|------|
| **DataLoader** | Loads, cleans, preprocesses data | 6 |
| **FeatureEngineering** | Creates 600+ features, selects best 150 | 6 |
| **ReturnPredictor** | LightGBM model for return prediction | 6 |
| **RiskForecaster** | LightGBM model for risk prediction | 6 |
| **QuantileBinningMapper** | Maps (r_hat, sigma_hat) → allocation | 6, 7 |
| **predict()** | Main inference function | 7 |
| **inference_server** | Kaggle evaluation server | 8, 9 |

### Model Architecture:

- **Algorithm**: LightGBM (gradient boosting)
- **Return Features**: ~150 selected features
- **Risk Features**: ~150 selected features
- **CV Strategy**: 5-fold time series walk-forward
- **Targets**: Forward returns (return model), Future volatility (risk model)
- **Position Strategy**: Quantile Binning (7 bins)
- **Output**: Allocation (0.0 to 2.0)

### Logging Configuration:

**Clean Notebook Logs** (Cell 3 설정):
- ✅ Notebook-level progress only: `[1/8] Loading data...`
- ❌ Suppressed verbose logs: Internal module logs hidden
- 🎯 Clear pipeline tracking: Step-by-step progress

```python
# Logging levels set to WARNING for all internal modules
logging.getLogger('src.data').setLevel(logging.WARNING)
logging.getLogger('src.features').setLevel(logging.WARNING)
# ... etc
```

### Performance Expectations:

Based on local optimization:
- **Return OOF Score**: ~0.012 - 0.015 (R² equivalent)
- **Risk OOF RMSE**: ~0.008 - 0.012 (volatility prediction)
- **Volatility Ratio**: ~1.1 - 1.2 (within constraint)
- **Leverage**: Minimal (mostly < 10%)

### 🎯 Execution Flow:

1. **Cell 1-5**: Setup (imports, config)
2. **Cell 6**: 🔥 **MAIN TRAINING** (~5-10 min)
3. **Cell 7**: Define predict() function
4. **Cell 8**: Initialize inference server
5. **Cell 9**: 
   - **Kaggle**: Start server (waits for real-time data)
   - **Local**: Generate sample predictions for testing

### 🎯 Ready for Submission!

1. ✅ ZIP created: `prediction_market_modules.zip`
2. ✅ Upload to Kaggle Datasets
3. ✅ Add dataset to this notebook
4. ✅ Update DATASET_NAME in Cell 3
5. ✅ Click "Run All"
6. ✅ Wait ~5-10 minutes for training
7. ✅ Submit to competition!

**Good luck! 🚀**